### **Feistel Network**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Feistel_Cipher.ipynb)

> A **Feistel network** splits a block into two halves and, in each round, mixes one half with a **round function** of the other half and a round key. Its key property: **decryption reuses the same structure with the round keys in reverse order**, even when the round function cannot be inverted. DES and many other block ciphers are Feistel networks.

> [!WARNING]
> Educational code. Never use these implementations to protect real data: use a vetted library (for example, `cryptography`) with modern, authenticated algorithms.

#### **01. Structure**

For each round *i* with key *Ki*:

* `L(i+1) = R(i)`
* `R(i+1) = L(i) XOR F(R(i), Ki)`

Decryption applies the same steps with the keys in reverse order. Here, `F` is HMAC-SHA256 truncated to the half-block size: a one-way, non-linear function.

In [1]:
# A generic Feistel network over 16-byte blocks
import hashlib
import hmac
import secrets

BLOCK = 16
HALF = BLOCK // 2

def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

def round_function(half, key):
    return hmac.new(key, half, hashlib.sha256).digest()[:HALF]   # one-way: never inverted

def feistel(block, round_keys, F=round_function):
    left, right = block[:HALF], block[HALF:]
    for key in round_keys:
        left, right = right, xor(left, F(right, key))
    return right + left            # final swap, so decryption uses the same function

def encrypt_block(block, round_keys, F=round_function):
    return feistel(block, round_keys, F)

def decrypt_block(block, round_keys, F=round_function):
    return feistel(block, list(reversed(round_keys)), F)

round_keys = [secrets.token_bytes(16) for _ in range(8)]   # 8 rounds, random 128-bit keys
block = b"Crypto in Python"                               # exactly 16 bytes
ciphertext = encrypt_block(block, round_keys)
print("Plaintext :", block)
print("Ciphertext:", ciphertext.hex())
print("Decrypted :", decrypt_block(ciphertext, round_keys))
assert decrypt_block(ciphertext, round_keys) == block

Plaintext : b'Crypto in Python'
Ciphertext: 455393e016eaa8ff8cf52283feab6939
Decrypted : b'Crypto in Python'


#### **02. Avalanche Effect**

A good cipher changes about half of the output bits when a single input bit changes. With few rounds, the change stays confined to one half.

In [2]:
# Count how many of the 128 output bits change when 1 plaintext bit is flipped
def bit_difference(a, b):
    return sum(bin(x ^ y).count("1") for x, y in zip(a, b))

flipped = bytes([block[0] ^ 0x01]) + block[1:]
print(f"{'rounds':>6} | {'bits changed (of 128)':>21}")
for rounds in (1, 2, 3, 4, 8):
    keys = round_keys[:rounds]
    print(f"{rounds:>6} | {bit_difference(encrypt_block(block, keys), encrypt_block(flipped, keys)):>21}")

rounds | bits changed (of 128)
     1 |                     1
     2 |                    39
     3 |                    72
     4 |                    69
     8 |                    64


#### **03. Why the Round Function Matters**

The original version of this notebook used `F(R, K) = R XOR K`, a **linear** function. With a linear `F`, the whole cipher becomes a XOR with values derived from the keys, so **one known plaintext-ciphertext pair breaks every other message**.

In [3]:
# Known-plaintext attack against a Feistel network with a linear round function
def linear_F(half, key):
    return xor(half, key[:HALF])

weak_keys = [secrets.token_bytes(16) for _ in range(2)]
known_plain = b"KNOWN PLAINTEXT!"
known_cipher = encrypt_block(known_plain, weak_keys, linear_F)

# For 2 rounds with linear F: output = (L xor k2, L xor R xor k1), so the key material is recoverable
target_plain = b"top secret value"
target_cipher = encrypt_block(target_plain, weak_keys, linear_F)

def attack(cipher_block):
    kl = xor(known_cipher[:HALF], known_plain[:HALF])                                # = k1 XOR k2
    kr = xor(known_cipher[HALF:], xor(known_plain[:HALF], known_plain[HALF:]))       # = k1
    left = xor(cipher_block[:HALF], kl)
    right = xor(xor(cipher_block[HALF:], kr), left)
    return left + right

print("Recovered without the keys:", attack(target_cipher))
assert attack(target_cipher) == target_plain

Recovered without the keys: b'top secret value'


#### **Key Takeaways**

* Feistel decryption works for any round function, which is why `F` can be one-way.
* Security comes from a non-linear round function and enough rounds (DES uses 16).
* A linear round function turns the cipher into a XOR, broken by a single known plaintext.